# 网络参数集的插值

通常，在改变其他参数（如温度、电压、电流等）时，会记录一组 `Networks`。一旦获取了这组数据，有时需要估计网络在测量参数值之间的行为。为此，可以对一组网络进行插值，以获得插值后的网络。本示例演示了如何使用 [NetworkSets](../../tutorials/NetworkSet.ipynb) 来实现这一点。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf

rf.stylely()

## Narda 3752 相移器

在本例中，我们将在 1.5 GHz 频率下对一个旧型号的 [narda 相移器 3752](https://nardamiteq.com/docs/119-PHASESHIFTERS.PDF) 进行测试。![narda 3752 相移器](phase_shifter_measurements/Narda_3752.jpg) :为了确定在该特定频率下可以获得的相移值，我们测量了 1-2 GHz 频段内相移旋钮的 19 个位置（从 0 到 180）的散射参数。这些测量数据被加载到一个 [NetworkSets](../../tutorials/NetworkSet.ipynb) 对象中：

In [ ]:
# Array containing the 19 phase shift indicator values
indicators_mes = np.linspace(0, 180, num=19)  # from 0 to 180 per 10

In [ ]:
ntw_set = rf.networkSet.NetworkSet.from_zip('phase_shifter_measurements/phase_shifter_measurements.zip')
print('ntw_set contains', len(ntw_set), 'networks')

我们从网络参数中提取在特定频率 1.5 GHz 处的相位偏移和 $S_{11}$。

In [ ]:
f = '1.5 GHz'
phases_mes = np.squeeze([ntw[f].s21.s_deg for ntw in ntw_set])
s11_mes = np.squeeze([ntw[f].s11.s_db for ntw in ntw_set])

然而，我们希望获得该移相器的中间设置对应的相位偏移值。为此，让我们通过插值测量到的网络数据来创建一个新的网络。

In [ ]:
# the indicator values for which we want to interpolate the network
indicators = np.linspace(0, 180, num=181)  # every degrees for 0 to 180

phases_interp = [ntw_set.interpolate_from_network(indicators_mes, phi)['1.5GHz'].s21.s_deg for phi in indicators]
phases_interp = np.squeeze(phases_interp)

s11_interp = [ntw_set.interpolate_from_network(
    indicators_mes, phi, interp_kind='quadratic')['1.5GHz'].s11.s_db for phi in indicators]
s11_interp = np.squeeze(s11_interp)

print('We have interpolated the network for', len(phases_interp), 'points')

现在让我们将所有内容绘制成图。

In [ ]:
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(10,7), sharex=True)
ax1.set_title('Narda Phase Shifter vs Indicator @ 1.5 GHz', fontsize=10)
ax1.plot(indicators, s11_interp, label='Interpolated network')
ax1.plot(indicators_mes, s11_mes, '.', ms=10, label='Measurements')
ax1.legend()
ax1.set_ylabel(r'$S_{11}$ [dB]')

ax2.plot(indicators, phases_interp, label='Interpolated network')
ax2.plot(indicators_mes, phases_mes, '.', ms=10, label='Measurements')
ax2.set_xlabel('Phase shifter indicator')
ax2.set_ylabel('$S_{21}$ [deg]')

ax2.fill_between(indicators, phases_mes[0], phases_mes[-1], alpha=0.2, color='r')
ax2.text(40, 25, 'not available zone')
ax2.legend()
ax2.set_ylim(-200, 200)

fig.tight_layout()